# 02 — Statistical Analysis
Calculate descriptive statistics, distribution diagnostics and correlations. Confirm metric semantics before interpretation.

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv("../.env")
host = os.getenv("PGHOST", "localhost")
port = os.getenv("PGPORT", "5432")
database = os.getenv("PGDATABASE", "ecommerce_practice")
user = os.getenv("PGUSER", "postgres")
password = os.getenv("PGPASSWORD", "")
schema = os.getenv("PGSCHEMA", "day24_gold")

if not password:
    raise ValueError("Set PGPASSWORD in the project-root .env file.")

engine = create_engine(
    f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"
)
print("Connected to PostgreSQL")


In [ ]:
fact = pd.read_sql(text(f'SELECT * FROM "{schema}"."fact_sales"'), engine)
numeric = fact.select_dtypes(include="number")
display(numeric.describe().T)


## Correlation analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric.shape[1] >= 2:
    corr = numeric.corr(numeric_only=True)
    plt.figure(figsize=(11, 8))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0)
    plt.title("Numeric Field Correlation")
    plt.tight_layout()
    plt.show()
else:
    print("At least two numeric columns are needed for correlation.")


## Outlier screening using IQR

In [ ]:
outlier_summary = []
for col in numeric.columns:
    q1, q3 = numeric[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = ((numeric[col] < lower) | (numeric[col] > upper)).sum()
    outlier_summary.append({
        "column": col, "lower_bound": lower,
        "upper_bound": upper, "potential_outliers": int(count)
    })
display(pd.DataFrame(outlier_summary))


## Statistical caution
A correlation is not evidence of causation. Investigate data types, sample size, skew and business definitions before presenting conclusions.